# Prompt compression — cut token usage on structured prompts

Aura can compress the *input* side of a request before it hits the model:

- **TOON** — Token-Oriented Object Notation, best for JSON arrays / uniform data
- **YAML** — fewer delimiters for nested objects
- **AISP** — symbolic notation for math-heavy content
- **JSON minify** — whitespace removal + key shortening

`auto_select` picks the best strategy per content type, and `target_ratio` asks for a target
compression factor (0.4 ≈ 60% fewer tokens).

This notebook sends one long structured prompt twice — plain and compressed — and compares
the billed input tokens.

In [ ]:
from aura import AuraClient

client = AuraClient()

In [ ]:
# A realistic structured payload: an order with a line-item array.
order_prompt = (
    "You are an order validator. Check the following order and report any "
    "discrepancies between the line items and the totals.\n\n"
    "ORDER:\n"
    "{\n"
    "  \"order_id\": \"ORD-78412\",\n"
    "  \"customer\": {\"name\": \"Aarav Mehta\", \"tier\": \"gold\"},\n"
    "  \"items\": [\n"
    "    {\"sku\": \"A-101\", \"name\": \"Wireless Mouse\", \"qty\": 2, \"unit_price\": 24.99},\n"
    "    {\"sku\": \"B-220\", \"name\": \"Mechanical Keyboard\", \"qty\": 1, \"unit_price\": 89.50},\n"
    "    {\"sku\": \"C-330\", \"name\": \"USB-C Hub 7-in-1\", \"qty\": 3, \"unit_price\": 39.00},\n"
    "    {\"sku\": \"D-441\", \"name\": \"Laptop Stand\", \"qty\": 1, \"unit_price\": 54.25}\n"
    "  ],\n"
    "  \"subtotal\": 319.23,\n"
    "  \"tax_rate\": 0.18,\n"
    "  \"shipping\": 0.00,\n"
    "  \"discount\": 15.00\n"
    "}\n\n"
    "List each discrepancy and the corrected totals."
)

In [ ]:
# 1. Baseline: same prompt, no compression
baseline = client.responses.create(model="gpt-5.4-mini", input=order_prompt)
b_in = baseline.usage.input_tokens if baseline.usage else 0
print(f"baseline input tokens: {b_in}")

In [ ]:
# 2. Same prompt, compression enabled (auto-select strategy)
compressed = client.responses.create(
    model="gpt-5.4-mini",
    input=order_prompt,
    compression={
        "enabled": True,
        "auto_select": True,
        "target_ratio": 0.4,  # aim for ~60% fewer input tokens
    },
)
c_in = compressed.usage.input_tokens if compressed.usage else 0
print(f"compressed input tokens: {c_in}")

In [ ]:
# 3. Savings table
if b_in and c_in:
    pct = 100.0 * (b_in - c_in) / b_in
    print(f"{'':28} {'tokens':>8} {'saved':>8}")
    print(f"{'baseline':28} {b_in:>8} {'—':>8}")
    print(f"{'compressed (auto_select)':28} {c_in:>8} {f'{pct:.0f}%':>8}")
else:
    print("usage metadata missing — check that your gateway returns usage.")

`compression` is an Aura extension on top of the Open Responses API — it rides in the
request body via extra kwargs. `auto_select` + `target_ratio` is the zero-config way to
start; power users can pin `data_format` (e.g. `"toon"`) or set `token_budget` instead.

Next: [05 — validation](05_validation.ipynb) for best-of-N / self-consistency.